<a href="https://colab.research.google.com/github/TikhonovEugeneA/Cloud-technologies/blob/main/EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Практическая работа 4</h3>

Разведочный анализ данных (exploratory data analysis, EDA)

ФИО, группа

# Импорт данных, форматы

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

импорт БД задачи "ирисы Фишира" из пакета seaborn

In [ ]:
data_iris = sns.load_dataset('iris')

display(data_iris.head(3))
display(data_iris.tail(3))

Проверка форматов и наличия пропусков в данных

In [ ]:
data_iris.info()

**Задание:**

опишите в текстовой ячейке ниже, какие форматы данных у признаков (типы данных), есть ли пропуски и в каких столбцах?

**Задание:**

создайте списки числовых и категориальных признаков набора данных (указать имена столбцов ( ['name1', 'name2', ...] )


In [ ]:
numerical_variables = [ ]
categorical_variables = [ ]

Описательная статистика для числовых данных

In [ ]:
data_iris[numerical_variables].describe()

Описательная статистика для категориальных данных

In [ ]:
data_iris[categorical_variables].describe()

# Визуализация разброса и распределений данных

## числовые данные

Пример для первой переменной "sepal_length"

In [ ]:
sns.boxplot(data = data_iris, x = numerical_variables[0])
plt.show()

Группировка данных по классам (группам) из другой переменной. В задаче с ирисами - это естественная групировка по сортам ирисов.

In [ ]:
sns.boxplot(data = data_iris, x = numerical_variables[0], y = 'species')
plt.show()

**Задание:**

Постройте диаграммы boxplot для остальных числовых переменных как в примере выше.

## категориальные данные

посчитать сколько раз встречается какая из категорий

In [ ]:
data_iris['species'].value_counts()

In [ ]:
data_iris['species'].value_counts().plot(kind='bar')
plt.show()

## гистограммы

Пример для первой переменной "sepal_length"

In [ ]:
sns.histplot(data = data_iris, x = numerical_variables[0], stat = 'probability')
plt.show()

Тоже самое с оценкой плотности распределения

In [ ]:
sns.histplot(data = data_iris, x = numerical_variables[0], stat = 'probability', kde = True)
plt.show()

Разбивка по группам

In [ ]:
sns.histplot(data = data_iris, x = numerical_variables[0], stat = 'probability', kde = True, hue = 'species')
plt.show()

**Задание:**

Постройте гистограмммы для остальных числовых переменных как в примере выше.

## парные графики переменных

In [ ]:
sns.pairplot(data = data_iris)
plt.show()

**Задание**:

Постройте аналогичный график с разбивкой по группам (параметр hue).

# Корреляции

## табличное представление

In [ ]:
corr_data = data_iris[numerical_variables].corr()
corr_data

Покажите в таблице:
* сильные корреляции (больше $0.8$ и меньше $-0.8$)
* средние корреляции (в интервалах $[0.4, 0.8]$ и $[-0.8, -0.4]$)

In [ ]:
# пример data[data > some_number and data < another_number]


## визуализация (heatmap)

In [ ]:
plt.figure()
# plt.figure(figsize=(X, Y)) X, Y - в дюймах чтобы картинка была читаемой
ax = sns.heatmap(corr_data, annot=True, vmax=1, vmin=-1, cmap='RdYlGn')
plt.show()

Постройте аналогичный график:
* только для сильных корреляций
* только для средних корреляций

# Парные графики для сильных корреляций

сделать для каждой "сильной" пары

In [ ]:
# var_1 = ..., var_2 = ...

sns.pairplot(data=data_iris,
                x_vars=var_1,
                y_vars=var_2,
                plot_kws={
                            'line_kws':{'color':'orange'},
                            'scatter_kws': {'color': 'g', 's':5}
                         },
                diag_kind='kde', kind='reg', height=5, aspect=10/5)

Сделать аналогичные графики с разбивкой по классам (сортам ирисов)

# Заполнение пропусков

## Смоделируем пропуски в данных (в релальных БД, в среднем до 10% данных оказываются незаполенными)

In [ ]:
# data.iat[row_number, col_number]
for j in range(4):
  k = np.random.randint(5,20)
  indexes = np.random.choice(150, k, replace=False)
  for i in indexes:
    data_iris.iat[i, j] = np.nan

In [ ]:
data_iris.info()

Опишите сколько пропусков в каком столбце.

## заполнение средним и медианой

заполните пропуски средним и медианой по каждому столбцу

In [ ]:
# data_fillna_mean = data.fillna(value = ...)
# data_fillna_median = data.fillna(value = ...)

## заполнение с учетом класса

заполните пропуски средним и медианой по каждому столбцу с учетом для какого класса найден пропуск (средние и медианы должны быть посчитаны для каждого класса)

# Снижение размерности, метод главных компонент

In [ ]:
# импорт метода МГК
from sklearn.decomposition import PCA

In [ ]:
pca = PCA()
pca_data = data_fillna_median[numerical_variables].copy()
pca.fit(pca_data)

In [ ]:
print(pca.explained_variance_ratio_*100)

* Какой процент дисперсии объясняет 1я главная компонента?
* Какой процент дисперсии объясняют 1я и 2я главные компоненты?

In [ ]:
# сохраним ГК в новый датафрейм
pca_df = pd.DataFrame(pca.transform(pca_data))
pcaNames = []
for i in range(4):
  pcaNames.append('pc'+str(i+1))
pca_df.columns = pcaNames
pca_df['species'] = data_iris['species'].copy()
pca_df.head(3)

Постройте график рассеяния (scatter plot) для первых 2х компонент с разбивкой по классам.

In [ ]:
# sns.scatterplot(data = , x = , y = , hue = )


Сделайте вывод. Удалось ли в меньшей размерности получить разделимые классы?